# Phase B: Spark MLlib — Arrest Prediction
# Tasks 5-7 Author: Mohamad Al Deri (ID: 230151)

In [1]:
!pip install pyspark pandas matplotlib

In [2]:
import random
from pyspark.sql import SparkSession
from pyspark.sql import Row

# 1. Start Spark Session
spark = SparkSession.builder.appName("LocalTest").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 2. Generate the 'df' dataset
crime_profiles = {"NARCOTICS": 0.85, "BATTERY": 0.30, "THEFT": 0.10}
districts = list(range(1, 26))

def generate_row():
    crime_type = random.choice(list(crime_profiles.keys()))
    arrest = random.random() < crime_profiles[crime_type]
    return Row(District=random.choice(districts), Primary_Type=crime_type,
               Date="01/01/2023 12:00:00 AM", Domestic=False, Arrest=arrest)

df = spark.createDataFrame([generate_row() for _ in range(10000)])
df = df.withColumnRenamed("Primary_Type", "Primary Type")

print(f"Success! Spark is running and generated {df.count()} rows in 'df'.")

Success! Spark is running and generated 10000 rows in 'df'.


In [3]:
# ============================================
# Task 5: Feature Engineering Pipeline
# Author: Mohamad Al Deri (ID: 230151)
# ============================================
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.functions import col, hour, to_timestamp

# Sample 5% of data
df_sampled = df.sample(fraction=0.05, seed=42)

# Pre-process
df_ml = df_sampled.withColumn("label", col("Arrest").cast("integer")) \
                  .withColumn("PrimaryType", col("Primary Type")) \
                  .withColumn("Domestic_str", col("Domestic").cast("string")) \
                  .withColumn("Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))) \
                  .dropna(subset=["District", "PrimaryType", "Hour", "Domestic_str", "label"])

# Transformers
type_indexer = StringIndexer(inputCol="PrimaryType", outputCol="crime_index", handleInvalid="skip")
domestic_indexer = StringIndexer(inputCol="Domestic_str", outputCol="domestic_index", handleInvalid="skip")
assembler = VectorAssembler(inputCols=["District", "crime_index", "Hour", "domestic_index"], outputCol="features")

# Split
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
train_df.cache()

print(f"Training set size: {train_df.count()} rows")
print(f"Testing set size: {test_df.count()} rows")

Training set size: 439 rows
Testing set size: 85 rows
